### Cloning Github REPO

In [10]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -3
print("\nRepo dir:", REPO_DIR)

/content/context-parametric-inversion-research
66deff0 (HEAD -> dev, origin/dev) Start mechanistic-analysis phase: flip-set builder + checkpoint phase map
9123e48 notebook: add judge-escalation pass (section 4b) for the base-model AMBIG items
b41b90b Merge master (--judge-prompt-version fix) into dev

Repo dir: /content/context-parametric-inversion-research


In [11]:
%cd {REPO_DIR}
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull --ff-only origin {BRANCH}
!git log --oneline -3


/content/context-parametric-inversion-research
From https://github.com/GIRIAYUSH/context-parametric-inversion-research
 * branch            dev        -> FETCH_HEAD
Already on 'dev'
Your branch is up to date with 'origin/dev'.
From https://github.com/GIRIAYUSH/context-parametric-inversion-research
 * branch            dev        -> FETCH_HEAD
Already up to date.
66deff0 (HEAD -> dev, origin/dev) Start mechanistic-analysis phase: flip-set builder + checkpoint phase map
9123e48 notebook: add judge-escalation pass (section 4b) for the base-model AMBIG items
b41b90b Merge master (--judge-prompt-version fix) into dev


In [3]:
import yaml, os, shutil

with open("src/mechanistic-analysis/checkpoints.yaml") as f:
    cfg = yaml.safe_load(f)
#setup staging directory for checkpoints in colab disk
STAGE_DIR = '/content/staged_checkpoints'

for run, run_cfg in cfg["runs"].items():
    drive_dir = run_cfg["drive_checkpoint_dir"]
    for phase, ckpt_name in run_cfg["phases"].items():
        if ckpt_name is None:
            continue  # baseline (no adapter) or missing phase (e.g. alpaca recovery)
        src = os.path.join(drive_dir, ckpt_name)
        dst = os.path.join(STAGE_DIR, run, ckpt_name)
        if os.path.isdir(dst):
            print(f"already staged: {run}/{ckpt_name}")
            continue
        print(f"staging {run}/{phase} -> {ckpt_name} ...")
        shutil.copytree(src, dst)

staging alpaca/peak -> checkpoint-100 ...
staging alpaca/trough -> checkpoint-800 ...
staging tulu/peak -> checkpoint-500 ...
staging tulu/trough -> checkpoint-3150 ...
staging tulu/recovery -> checkpoint-5000 ...


### Verify if staging is complete

In [12]:
print("\nDone. Staged checkpoints:")
!find {STAGE_DIR} -maxdepth 2 -type d


Done. Staged checkpoints:
/content/staged_checkpoints
/content/staged_checkpoints/alpaca
/content/staged_checkpoints/alpaca/checkpoint-100
/content/staged_checkpoints/alpaca/checkpoint-800
/content/staged_checkpoints/tulu
/content/staged_checkpoints/tulu/checkpoint-5000
/content/staged_checkpoints/tulu/checkpoint-500
/content/staged_checkpoints/tulu/checkpoint-3150


In [13]:
STAGE_DIR = '/content/staged_checkpoints'
from safetensors.torch import load_file
import json,os,re
import torch
import yaml
def load_deltas(checkpoint_path):
    """Returns {(layer_idx, proj_name)"""
    if checkpoint_path is None:
        return {}

    weights = load_file(os.path.join(checkpoint_path, "adapter_model.safetensors"))
    cfg = json.load(open(os.path.join(checkpoint_path, "adapter_config.json")))
    scaling = cfg["lora_alpha"] / cfg["r"]

    # group lora_A / lora_B pairs by (layer_idx, proj_name)
    pairs = {}
    for key, tensor in weights.items():
        m = re.search(r"layers\.(\d+)\.(\w+)\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)\.lora_(A|B)\.weight", key)
        if not m:
            continue
        layer_idx, _, proj_name, ab = m.groups()
        pairs.setdefault((int(layer_idx), proj_name), {})[ab] = tensor

    deltas = {}
    for (layer_idx, proj_name), ab in pairs.items():
        delta = (ab["B"] @ ab["A"]) * scaling
        deltas[(layer_idx, proj_name)] = delta
    return deltas


### Computing Delta Norm

In [14]:
def delta_norms(deltas):
    """{(layer_idx, proj_name): frobenius_norm}, 0.0 for anything missing."""
    return {k: torch.linalg.norm(v).item() for k, v in deltas.items()}

### Magnitude using Forbeius Norm

In [18]:
def diff_norms(deltas_a, deltas_b):
    keys = set(deltas_a) | set(deltas_b)
    out = {}
    for k in keys:
        if k in deltas_a and k in deltas_b:
            out[k] = torch.linalg.norm(deltas_b[k] - deltas_a[k]).item()
        elif k in deltas_b:
            out[k] = torch.linalg.norm(deltas_b[k]).item()   # baseline side has zero delta
        else:
            out[k] = torch.linalg.norm(deltas_a[k]).item()
    return out


In [19]:
cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))

In [20]:
def phase_path(run, phase):
    ckpt = cfg["runs"][run]["phases"][phase]
    return None if ckpt is None else os.path.join(STAGE_DIR, run, ckpt)

for run in ["alpaca", "tulu"]:
    print(f"=== {run} ===")
    phases = ["baseline", "peak", "trough", "recovery"]
    loaded = {p: load_deltas(phase_path(run, p)) for p in phases if phase_path(run, p) or p == "baseline"}

    pairs = [("baseline", "peak"), ("peak", "trough"), ("trough", "recovery")]
    for a, b in pairs:
        if a not in loaded or b not in loaded:
            print(f"  {a} -> {b}: skipped (missing phase for {run})")
            continue
        d = diff_norms(loaded[a], loaded[b])
        attn = sum(v for (_, proj), v in d.items() if proj in ("q_proj","k_proj","v_proj","o_proj"))
        mlp  = sum(v for (_, proj), v in d.items() if proj in ("gate_proj","up_proj","down_proj"))
        print(f"  {a} -> {b}:  attn_total={attn:.3f}   mlp_total={mlp:.3f}   ratio(mlp/attn)={mlp/attn:.3f}")

=== alpaca ===


: 

: 

: 